> 📒 이 노트북은 [펭귄 데이터로 배우는 인공지능](https://yanghyeokjin33.github.io/Algorithm/) 수업의 실습이에요. 원본: [https://github.com/YANGHYEOKJIN33/Algorithm](https://github.com/YANGHYEOKJIN33/Algorithm)

# 🤖 k-최근접 이웃 (k-Nearest Neighbors)

**분류 · 지도학습** — 새 펭귄과 가장 가까운 훈련 펭귄 k마리를 찾아 다수결로 종을 정한다.

    거리목록 ← 빈 리스트
    반복: 훈련 데이터의 각 펭귄 p
        거리목록에 (새 펭귄과 p의 거리, p의 종) 추가
    거리목록을 거리 순으로 정렬
    이웃 ← 앞에서 k개 → 세기표로 투표 → 가장 많은 종

> **Colab 사용법** — 셀 왼쪽의 ▶ 단추(또는 **Shift + Enter**)를 누르면 그 셀이 실행돼요.
> 위에서부터 **차례로** 실행하세요. 고쳐 보고 싶으면 먼저 **파일 → 드라이브에 사본 저장**을 눌러 내 것으로 만드세요.
> 파이썬을 몰라도 괜찮아요. 코드 위의 설명과 사이트의 의사코드를 짝지어 읽으면 돼요.

- Counter는 사이트의 "세기표(사전)"를 만들어 주는 도구예요.

In [ ]:
import pandas as pd
from collections import Counter
df = pd.read_csv('https://raw.githubusercontent.com/YANGHYEOKJIN33/Algorithm/main/data/penguins_clean.csv')
print(df.shape)

## 1. 의사코드 그대로 — 사이트와 같은 18마리로

- knn_predict 함수의 줄 하나하나가 사이트 ② 쪽 의사코드의 줄과 짝이에요.
- ** 0.5는 제곱근(√)이에요.
- k=1이면 가장 가까운 아델리 한 마리만 보고 틀리지만, k=3이면 턱끈 2표로 맞혀요(정답: 턱끈).

In [ ]:
train_ids = [1, 6, 14, 21, 31, 38, 277, 278, 280, 282, 285, 288, 153, 154, 158, 160, 161, 164]
train = df[df['번호'].isin(train_ids)][['부리길이', '부리깊이', '종']].values.tolist()
new = [43.5, 18.1]      # 341번 펭귄 (종을 모른다고 하자)

def knn_predict(train, new, k):
    dists = []                                             # 거리목록
    for p in train:
        d = ((new[0] - p[0])**2 + (new[1] - p[1])**2) ** 0.5   # 두 점 사이 거리
        dists.append((d, p[2]))                            # (거리, 종)
    dists.sort()                                           # 가까운 순으로
    neighbors = dists[:k]                                  # 앞에서 k개
    votes = Counter(label for d, label in neighbors)       # 세기표
    return votes.most_common(1)[0][0], neighbors

for k in [1, 3, 5]:
    pred, nb = knn_predict(train, new, k)
    print(f'k={k}: {pred}   이웃 = {[(round(d, 2), s) for d, s in nb]}')

## 2. scikit-learn으로 — 프로젝트에서 쓰는 방법

만들기 → `fit`(학습) → `predict`(예측) 세 줄이면 돼요.

- n_neighbors가 k예요.
- k-최근접 이웃의 fit()은 계산 없이 훈련 데이터를 기억해 둘 뿐이에요. 진짜 일은 predict()에서 거리를 재며 일어나요.
- 정확도 = 맞힌 개수 ÷ 테스트 개수

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

X = df[['부리길이', '부리깊이']]
y = df['종']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = KNeighborsClassifier(n_neighbors=3)   # k = 3
model.fit(X_train, y_train)                   # 학습 = 훈련 데이터를 기억해 두기
pred = model.predict(X_test)                  # 테스트 데이터 예측
print('정확도:', accuracy_score(y_test, pred))

- 이번에는 아델리로 예측해요! kneighbors()로 이웃 3마리를 꺼내 보면 까닭이 보여요.
- 341번은 272줄 훈련 데이터 안에 들어 있어요. 그래서 가장 가까운 이웃은 자기 자신(턱끈, 거리 0)이에요.
- 자기 자신도 한 표지만 100번·130번 아델리 2표에 져서 아델리로 예측해요. 341번은 아델리와 턱끈의 경계에 있는 펭귄이라 이웃에 따라 답이 흔들려요.
- 그래서 한 마리가 아니라, 학습에 쓰지 않은 테스트 데이터 전체의 정확도로 모델을 평가해요.

In [ ]:
# 341번 펭귄을 sklearn으로도 맞혀 보기
new = pd.DataFrame([[43.5, 18.1]], columns=['부리길이', '부리깊이'])
print('예측:', model.predict(new))
dist, idx = model.kneighbors(new)                  # 가장 가까운 이웃 3마리의 거리와 자리
near = df.loc[X_train.index[idx[0]], ['번호', '종']]
near['거리'] = dist[0].round(2)
print(near)

## 3. k를 바꿔 보기

- k가 너무 작으면 한 이웃에 휘둘리고, 너무 크면 멀리 있는 펭귄까지 투표해요. 알맞은 k를 실험으로 찾아요.

In [ ]:
for k in [1, 3, 5, 7, 9, 15]:
    m = KNeighborsClassifier(n_neighbors=k).fit(X_train, y_train)
    print(f'k={k:2d}  정확도 {accuracy_score(y_test, m.predict(X_test)):.3f}')

## 4. 그림으로 보기

- 그래프에 한글(제목·축 이름)이 깨지지 않도록 글꼴 도구를 설치해요. 처음 한 번 1~2분 걸릴 수 있어요.

In [ ]:
!pip install -q koreanize-matplotlib
try:
    import koreanize_matplotlib  # 그래프에 한글 글꼴을 입혀 줘요
except ImportError:
    print('한글 글꼴을 불러오지 못했어요. 그래프 글자가 네모로 보일 수 있어요.')
import matplotlib.pyplot as plt

- 종마다 모양을 다르게 해서 점을 찍어요. 별(★)이 맞혀 볼 펭귄이에요.

In [ ]:
for sp, mark in [('아델리', 'o'), ('턱끈', '^'), ('젠투', 's')]:
    part = df[df['종'] == sp]
    plt.scatter(part['부리길이'], part['부리깊이'], marker=mark, label=sp, alpha=0.7)
plt.scatter([43.5], [18.1], marker='*', s=300, c='black', label='341번 펭귄')
plt.xlabel('부리길이 (mm)'); plt.ylabel('부리깊이 (mm)'); plt.legend()
plt.show()

## 🚀 도전

- 속성을 4개(부리길이·부리깊이·날개길이·몸무게)로 늘려 정확도를 견줘 보세요.
- 몸무게(수천 g)는 다른 속성(수십 mm)보다 숫자가 훨씬 커서 거리를 혼자 좌우해요. `from sklearn.preprocessing import StandardScaler`로 크기를 맞추면 어떻게 될까요?